In [ ]:
!pip install -q transformers datasets scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from google.colab import drive

MODEL_NAME = "indobenchmark/indobert-base-p2"
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
EPOCHS = 5
MAX_LENGTH = 128
RANDOM_STATE = 42

drive.mount('/content/drive')
path = '/content/drive/MyDrive/Skripsi 3,5 Tahun/skripsi_dataset/Revisi/DATASET_SIAP_LABEL_FINAL_CEK.csv'

df = pd.read_csv(path)
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label', 'text_slang_removal']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f"Total data bersih: {len(df)}")
print(f"Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")

df_train, df_temp = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['label'])
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=df_temp['label'])

print(f"\nData Split:")
print(f"  Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print(f"  Train dist: {df_train['label'].value_counts().sort_index().to_dict()}")
print(f"  Val dist:   {df_val['label'].value_counts().sort_index().to_dict()}")
print(f"  Test dist:  {df_test['label'].value_counts().sort_index().to_dict()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text_slang_removal"], padding="max_length",
                     truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(df_train[['text_slang_removal', 'label']])
val_ds = Dataset.from_pandas(df_val[['text_slang_removal', 'label']])
test_ds = Dataset.from_pandas(df_test[['text_slang_removal', 'label']])

train_tokenized = train_ds.map(tokenize_function, batched=True)
val_tokenized = val_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir='./logs',
    logging_steps=50,
    save_total_limit=2,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("\nMemulai Fine-tuning IndoBERT...")
trainer.train()

print("\nMengevaluasi Data Uji...")
preds = trainer.predict(test_tokenized)
pred_labels = np.argmax(preds.predictions, axis=-1)
true_labels = test_tokenized['label']

print("\n" + "="*50)
print("           HASIL EVALUASI MODEL")
print("="*50)
print(classification_report(true_labels, pred_labels,
                          target_names=['Negatif', 'Netral', 'Positif'],
                          digits=4))

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Netral', 'Positif'],
            yticklabels=['Negatif', 'Netral', 'Positif'],
            cbar_kws={'label': 'Jumlah'})
plt.xlabel('Prediksi Model', fontsize=12)
plt.ylabel('Label Sebenarnya', fontsize=12)
plt.title('Confusion Matrix - Analisis Sentimen Publik Terhadap Polri', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

trainer.save_model("./model_sentimen_polri")
tokenizer.save_pretrained("./model_sentimen_polri")

print("\nSELESAI!")
print("Confusion matrix tersimpan: confusion_matrix.png")
print("Model tersimpan di: ./model_sentimen_polri")
print("\nUntuk Bab IV: Gunakan classification_report dan confusion_matrix di atas")

Mounted at /content/drive
Total data bersih: 1000
Distribusi label: {0: 350, 1: 300, 2: 350}

Data Split:
  Train: 800 | Val: 100 | Test: 100
  Train dist: {0: 280, 1: 240, 2: 280}
  Val dist:   {0: 35, 1: 30, 2: 35}
  Test dist:  {0: 35, 1: 30, 2: 35}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (10 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.663600,0.458121,0.820000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 